In [5]:
# Jupyter Notebook cell (Python 3)
# ---------------------------------------------------------------------------
# This code demonstrates how to process an input video to detect
# and draw hand meshes (skeletons) with MediaPipe's Hand Landmarker.
# Each detected hand is drawn using two color codes:
#   1. One color for the circles/landmarks
#   2. A contrasting color for the skeletal connections/lines
#
# Instead of storing both colors in a single list of pairs, we now store them
# in two separate lists (CIRCLE_COLORS and CONNECTION_COLORS). For hand i,
# we select CIRCLE_COLORS[i] and CONNECTION_COLORS[i] (cycling through with %).
#
# Requirements:
#   - mediapipe (>= 0.9.1 or the latest version that supports 'tasks')
#   - opencv-python
#   - numpy
#   - (optional) plotly, if you want to visualize images interactively
#
# Usage:
#   1. Place your input video in the same folder or provide its full path.
#   2. Update 'input_video_path' below with your video file name or path.
#   3. Run this cell. The output video will be saved in the same directory
#      with the suffix "_handskel" added to the name.

import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import os

def process_video_with_hand_landmarker(input_video_path: str, 
                                       model_path: str = "/home/robotics/Downloads/hand_landmarker.task",
                                       max_num_hands: int = 4):
    """
    Processes the given video file to detect hand landmarks using MediaPipe's
    HandLandmarker (Tasks API). Each detected hand is drawn with two separate,
    contrasting color codes: one for circles (landmarks) and one for the
    skeletal connections.

    The result is saved to an output video file with '_handskel' appended
    to the original filename.

    Args:
        input_video_path (str): Path to the input video file.
        model_path (str): Path to the MediaPipe Hand Landmarker model.
                          Default is '/home/robotics/Downloads/hand_landmarker.task'.
        max_num_hands (int): Max number of hands to detect per frame.
    """

    # ----------------------------------------------------------------
    # 1. Prepare the output video path
    # ----------------------------------------------------------------
    base_name, ext = os.path.splitext(input_video_path)
    output_video_path = f"{base_name}_handskel{ext}"

    # ----------------------------------------------------------------
    # 2. Set up the video capture and writer
    # ----------------------------------------------------------------
    cap = cv2.VideoCapture(input_video_path)
    if not cap.isOpened():
        print(f"Error: Could not open video file {input_video_path}")
        return
    
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        fps = 30  # fallback if FPS cannot be retrieved
    
    # Choose appropriate codec
    if ext.lower() == '.mp4':
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    else:
        fourcc = cv2.VideoWriter_fourcc(*'XVID')
    
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))
    
    # ----------------------------------------------------------------
    # 3. Initialize the MediaPipe Hand Landmarker
    # ----------------------------------------------------------------
    BaseOptions = mp.tasks.BaseOptions
    HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
    RunningMode = mp.tasks.vision.RunningMode
    
    options = HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=model_path),
        running_mode=RunningMode.VIDEO,
        num_hands=max_num_hands
    )
    
    # Two separate lists for circle colors and connection colors
    # The i-th hand gets its color from CIRCLE_COLORS[i] and CONNECTION_COLORS[i].
    # If there are more hands than colors, we cycle through with %.
    CIRCLE_COLORS = [
        (255,   0,   0),  # Blue
        (  0, 255,   0),  # Green
        (  0,   0, 255),  # Red
        (255, 255, 255),  # White
        (128,   0, 128),  # Purple
        (255, 105, 180),  # Pink
        (  0, 255, 255),  # Yellow
        (255,   0, 255)   # Magenta
    ]
    
    CONNECTION_COLORS = [
        (  0, 255, 255),  # Yellow
        (255,   0, 255),  # Magenta
        (255, 255,   0),  # Cyan
        (  0,   0,   0),  # Black
        (128, 128,   0),  # Olive
        (105, 255, 180),  # Pastel Greenish
        (255,   0,   0),  # Blue
        (  0,   0, 255)   # Red
    ]
    
    # Standard MediaPipe hand connections (+ some palm connections).
    HAND_CONNECTIONS = [
        (0,1), (1,2), (2,3), (3,4),           # Thumb
        (0,5), (5,6), (6,7), (7,8),           # Index finger
        (0,9), (5,9), (9,10), (10,11), (11,12),     # Middle finger
        (0,13), (9,13), (13,14), (14,15), (15,16),  # Ring finger
        (0,17), (13,17), (17,18), (18,19), (19,20)  # Pinky
    ]
    
    with mp.tasks.vision.HandLandmarker.create_from_options(options) as hand_landmarker:
        frame_index = 0
        
        while True:
            success, frame_bgr = cap.read()
            if not success:
                break  # End of video

            # Convert BGR to RGB for MediaPipe
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

            # Create a MediaPipe Image object
            mp_image = mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=frame_rgb
            )

            # Calculate the frame timestamp in milliseconds
            frame_timestamp_ms = int((1000.0 / fps) * frame_index)

            # Detect hands in the frame
            detection_result = hand_landmarker.detect_for_video(
                mp_image, frame_timestamp_ms
            )
            
            # ----------------------------------------------------------------
            # 4. Draw landmarks on the frame using separate lists
            # ----------------------------------------------------------------
            if detection_result.hand_landmarks:
                for idx, landmarks in enumerate(detection_result.hand_landmarks):
                    circle_color     = CIRCLE_COLORS[idx % len(CIRCLE_COLORS)]
                    connection_color = CONNECTION_COLORS[idx % len(CONNECTION_COLORS)]

                    # Draw circles at each landmark
                    for landmark in landmarks:
                        x_px = int(landmark.x * width)
                        y_px = int(landmark.y * height)
                        # Radius=5, fill the circle
                        cv2.circle(frame_bgr, (x_px, y_px), 5, circle_color, -1, cv2.LINE_AA)
                    
                    # Draw lines for the skeleton
                    for connection in HAND_CONNECTIONS:
                        start_idx, end_idx = connection
                        x_start = int(landmarks[start_idx].x * width)
                        y_start = int(landmarks[start_idx].y * height)
                        x_end   = int(landmarks[end_idx].x * width)
                        y_end   = int(landmarks[end_idx].y * height)
                        cv2.line(frame_bgr, (x_start, y_start), (x_end, y_end),
                                 color=connection_color, thickness=2, lineType=cv2.LINE_AA)

            # Write the annotated frame
            out.write(frame_bgr)
            frame_index += 1
    
    # Clean up
    cap.release()
    out.release()
    print(f"Output video saved to: {output_video_path}")


# Example usage:
# input_video_path = "/home/robotics/Desktop/rl_2_p01.avi"
input_video_path = "/home/robotics/Desktop/web1_output_p01.avi"
process_video_with_hand_landmarker(input_video_path)


I0000 00:00:1741177010.653895    7797 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1741177010.708532    8848 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 560.35.05), renderer: NVIDIA GeForce RTX 4060 Ti/PCIe/SSE2
W0000 00:00:1741177010.723185    8865 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1741177010.727537    8871 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Output video saved to: /home/robotics/Desktop/web1_output_p01_handskel.avi
